# Lần 3a — scale 10K (chỉ Lean, $0, ~10 giờ)

**Không cần sửa gì trong notebook này** — cấu hình đã điền sẵn.

Trước khi bấm *Save Version → Save & Run All*:
1. **Settings:** Accelerator = *None* · Internet = *On*.
2. **Input:** chỉ cần dataset **`g6-errors-v1`** (Add Input → Your Work → Datasets).
   Không thêm input nào khác.
3. **Secrets:** không cần.

Kết quả: tải file `out_<phiên>_<giờ>.zip` ở tab **Output** khi chạy xong.

## CẤU HÌNH

In [ ]:
RUN_DATA     = False
N_SEEDS      = 5000
RUN_RQ_FREE  = False
RUN_PILOT    = False
RUN_LLM_MAIN = False
SCALE_TIERS  = ["errors_10k"]
SCALE_WORKERS, SCALE_REPEATS = [1, 2, 4], [1, 2, 3]
RUN_TAG      = ""
BRANCH = "feat/verifier-guided-pipeline"

## 0. Lấy code

In [ ]:
import os, glob
os.chdir("/root")
!rm -rf repo && git clone -q --depth 1 -b {BRANCH} https://github.com/tamdv/ai4math-verifier-guided-repair.git repo
os.chdir("/root/repo")
!git log --oneline -1
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]

import datetime, zoneinfo
_tag = RUN_TAG or "-".join(
    (["data"] if RUN_DATA else []) + (["free"] if RUN_RQ_FREE else []) + (["pilot"] if RUN_PILOT else [])
    + (["llm"] if RUN_LLM_MAIN else []) + (["scale-" + "-".join(t.replace("errors_", "") for t in SCALE_TIERS)] if SCALE_TIERS else [])
) or "session"
_stamp = datetime.datetime.now(zoneinfo.ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M")
OUT_ZIP = f"out_{_tag}_{_stamp}.zip"   # một file cho cả phiên, ghi đè sau mỗi bước
print("Kết quả phiên này sẽ lưu vào:", OUT_ZIP)

def save_out(stage):
    """Snapshot results into /kaggle/working/<OUT_ZIP> (survives a 12 h timeout)."""
    get_ipython().system(f"mkdir -p /kaggle/working/out/errors && cp -r results /kaggle/working/out/ 2>/dev/null; cp data/errors/*.jsonl /kaggle/working/out/errors/ 2>/dev/null; cp data/seeds/*.jsonl /kaggle/working/out/ 2>/dev/null; cp data_sample/errors_sample_100.jsonl /kaggle/working/out/ 2>/dev/null; cd /kaggle/working && rm -f {OUT_ZIP} && zip -qr {OUT_ZIP} out")
    print(f"[saved after {stage}] -> {OUT_ZIP}")

## 1. Cài môi trường (~10–15 phút)

In [ ]:
!bash scripts/kaggle_setup.sh
if RUN_PILOT or RUN_LLM_MAIN:   # 1 lần gọi rất nhỏ để kiểm tra key — chỉ khi phiên này có dùng LLM
    !python scripts/check_llm.py

## 2. Dữ liệu

In [ ]:
def lines(p):
    with open(p) as f: return sum(1 for _ in f)

def find_previous_data():
    """Locate a previous run's dataset under /kaggle/input at any depth (unpacked or zipped)."""
    hits = glob.glob("/kaggle/input/**/errors_10k.jsonl", recursive=True)
    if not hits:
        for z in glob.glob("/kaggle/input/**/out*.zip", recursive=True):
            dest = "/kaggle/temp/prev_" + str(abs(hash(z)))
            !mkdir -p {dest} && unzip -q -o "{z}" -d {dest}
            hits += glob.glob(dest + "/**/errors_10k.jsonl", recursive=True)
    return max(hits, key=lines) if hits else None

print("Inputs attached:")
!find /kaggle/input -maxdepth 4 | head -30
best = find_previous_data()
if best:
    err_dir = os.path.dirname(best)
    seed_dir = os.path.dirname(err_dir)
    print("Dùng lại dữ liệu từ:", err_dir, "| errors_10k =", lines(best), "dòng")
    !mkdir -p data/errors data/seeds && cp {err_dir}/*.jsonl data/errors/ && cp {seed_dir}/seeds_verified.jsonl {seed_dir}/natural_errors.jsonl data/seeds/ 2>/dev/null; true
elif RUN_DATA:
    !python scripts/build_seeds.py --n {N_SEEDS} --workers 4 --timeout 30
    !python scripts/check_strip_comments.py --workers 4 --limit 500
    !python scripts/generate_error_dataset.py --workers 4 --max-per-seed 5 --timeout 30
else:
    raise SystemExit("KHÔNG TÌM THẤY DỮ LIỆU: hãy Add Input -> output của version lần 2 rồi chạy lại.")
!wc -l data/errors/*.jsonl
assert os.path.exists("data/errors/errors_10k.jsonl"), "Thiếu data/errors/errors_10k.jsonl"
save_out("data")

## 3. Thí nghiệm miễn phí trên 1K hold-out

In [ ]:
if RUN_RQ_FREE:
    !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy none --budget 0 --workers 4 --tag floor
    !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy rule_only --budget 3 --workers 4 --tag rule
    save_out("rq free")

## 4. Pilot LLM (chỉ khi RUN_PILOT)

In [ ]:
if RUN_PILOT:
    !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --limit 50 --strategy blind --budget 1 --workers 4 --run-cap 0.10 --tag pilot
    !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --limit 50 --strategy llm_only --budget 1 --workers 4 --run-cap 0.10 --tag pilot
    save_out("pilot")

## 5. Thí nghiệm LLM chính

In [ ]:
if RUN_LLM_MAIN:
    !python scripts/estimate_cost.py --data data/errors/errors_10k.jsonl
    for r in [1, 2, 3]:   # lần 2-3 dùng cache câu trả lời => ~0đ
        for strat, b in [("blind", 1), ("llm_only", 1), ("hybrid", 1), ("llm_only", 3), ("hybrid", 3)]:
            !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy {strat} --budget {b} --workers 4 --repeat-index {r} --tag rq
        save_out(f"rq repeat {r}")
    !python scripts/run_experiment.py --data data/errors/errors_10k.jsonl --split holdout --strategy hybrid --budget 1 --workers 4 --run-cap 3.0 --tag scale-llm
    save_out("scale-llm")
    !python scripts/run_experiment.py --data data/errors/errors_1k.jsonl --split holdout --strategy llm_only --budget 1 --workers 4 --model gemini-3.6-flash --tag strong-model
    save_out("strong model")

## 6. Scalability chỉ dùng Lean ($0) — mỗi lần chạy lưu ngay

In [ ]:
for data in SCALE_TIERS:
    for r in SCALE_REPEATS:
        for w in SCALE_WORKERS:
            !python scripts/run_experiment.py --data data/errors/{data}.jsonl --split all --strategy rule_only --budget 1 --workers {w} --repeat-index {r} --tag scale
            save_out(f"scale {data} w{w} r{r}")

## 7. Tổng kết phiên

In [ ]:
save_out("end")
!ls results/raw | tail -40
!cat results/cost_ledger.jsonl 2>/dev/null | python -c "import sys,json; r=[json.loads(l) for l in sys.stdin]; print(f'Chi tiêu phiên này: ${sum(x[\"cost_usd\"] for x in r):.4f} ({len(r)} lần gọi)')" || echo "Không có lần gọi trả phí" 